<a href="https://colab.research.google.com/github/ankitf/streetworks-compliance/blob/main/barrier_continuity_assessment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Street Works Compliance using Cosmos Reason-2**
Decision Reel: Spatiotemporal Barrier Continuity Assessment

## **Recipe Overview**

This recipe demonstrates how to use **NVIDIA Cosmos Reason-2** to transform short, egocentric street-works video footage into a **Decision Reel** — a structured, time-ordered record of compliance decisions, supporting visual evidence, and natural-language rationale.

Given a natural-language compliance intent (for example, “Verify pedestrian safety and traffic separation”), the recipe applies Cosmos Reason-2’s spatiotemporal reasoning capabilities to analyze a 60-second mobile “walk-around” video of a work site. Rather than performing simple object detection, the model reasons over **physical relationships, temporal changes, and spatial constraints** defined by safety regulations.

The output is a **Decision Reel**:
a sequence of timestamped observations that includes:

* the detected compliance condition or deviation,
* the corresponding video evidence (frames or clips), and
* a structured explanation describing why the configuration is considered compliant or non-compliant.

This approach enables **remote, on-demand compliance auditing** for dynamic street-works environments, replacing continuous video streams with short, high-intent clips that are easier to review, cheaper to process, and suitable for audit trails. The recipe highlights how Cosmos Reason-2 can serve as a virtual compliance assistant—augmenting, rather than replacing, human inspectors by surfacing high-risk conditions with clear, inspectable reasoning.

## **Motivation for Street Works Compliance**

Street-works compliance is a particularly challenging and high-impact domain for intent-driven video reasoning.

Ensuring utility work sites (gas, water, fiber) adhere to strict safety standards like the [Red Book (UK)](https://assets.publishing.service.gov.uk/media/5a7d8038e5274a676d532707/safety-at-streetworks.pdf) or [MUTCD (US)](https://mutcd.fhwa.dot.gov) is critical but logistically difficult.

#### **The Operational Challenge: Scaling Expertise**

* **Safety Risk**: Missing barriers or misplaced cones lead to pedestrian incursions into hazardous zones.
* **Economic Impact**: Non-compliance results in heavy fines and project delays.
* **Logistical Bottleneck**: Most sites are transient and geographically dispersed. Sending certified auditors to every site is expensive, slow, and carbon-intensive.

#### **The Technical Gap: Beyond Object Detection**

Traditional computer vision pipelines (standard object detection) struggle in these dynamic, unconstrained environments:
* **Static vs. Dynamic**: They can identify a "cone" but cannot reason if that cone is placed correctly relative to a moving traffic lane or a shifting trench edge.
* **Model Fatigue**: Continuous monitoring with fixed cameras is data-heavy, energy-inefficient, and prone to "blind spots" caused by fixed angles or lighting.

#### **The Solution: AI-Augmented Remote Auditing**

We propose a **self-serve, remote-inspection workflow** that bridges the gap between field reality and regulatory oversight.

* **Field Capture** : A technician records a 60-second walk-around video of the site setup using a mobile device.

* **Cosmos Analysis**: The video is processed by NVIDIA Cosmos Reason, acting as a Virtual Auditor. The model uses its spatiotemporal reasoning to "read" the site, identifying physical relationships (e.g., barrier-to-traffic distance)  that simpler models miss.

* **Intelligent Triaging**: Instead of a simple "Pass/Fail," the model generates a Reasoned Summary of Observations.

* **Human Verification**: Any detected deviations or high-risk configurations are instantly flagged to a Human Operative. This expert reviews the reasoned explanations and the specific video frames, making the final call on whether to halt work or approve the site.

* **The "Decision Reel**: The final output is a structured Decision Reel—a time-ordered set of compliance decisions, evidence clips, and rationale. This serves as a "Digital Receipt" of safety, providing a clear audit trail for every site inspection, complete with natural-language evidence for why a specific setup was approved or rejected.

*This approach replaces high-volume, low-value constant streams with high-intent, short-duration video clips, significantly reducing compute costs and energy consumption. Additionally, by moving the requirement of specialised hardware and significant GPU compute to the cloud, it directly addresses the real-time edge constraints.*

## **Why NVIDIA Cosmos Reason-2?**

Environments are dynamic, assessments multi-variate and decisions require reasoning and understanding the corelation across multiple objects and rules. Simple object-detection models are insufficient to provide a full assessment, and are usually integrated into multiple-model pipelines that add the logic and reasoning layer. This is where Cosmos Reason-2, with its understanding of the "World Model" is a good fit.

* **Complex Spatiotemporal Reasoning**: Unlike simple detectors, Cosmos understands the physical correlation between multiple objects (e.g., the distance between a spoil heap and a trench) over time. Streetworks require precise geometric relationships. A model must not just see a "cone" and a "barrier," but understand if they are spaced exactly 0.5 meters apart per regulation.

* **Zero-Shot Regulation Logic**: By applying "physical common sense," the model can interpret "Red Book" standards—identifying not just what objects are present, but why their specific arrangement constitutes a violation.


## **Challenges for NVIDIA Cosmos Reason-2**

While Cosmos Reason-2 represents a leap in Physical AI, the dynamic envionment of this usecase presents several technical hurdles:

1. **Egocentric Motion & "Shaky Cam" Stability**:

    A field technician’s handheld "walk-around" video introduces significant ego-motion. The model must be able to see through the blur, handle perspective distortion and distinguish between a moving object (a car) and camera movement (the walking technician) to accurately reason about the site’s geometry. Maintaining spatial consistency while the camera is bouncing is a primary challenge for spatiotemporal models.

2. **Environmental Variability**:

    Unlike the consistent lighting of a manufacturing unit or a warehouse, streetworks occur in rain, snow, or at night with glare from emergency lights. Streetworks also change with traffic flow and Cosmos must reason about temporal risks—e.g., "A car is approaching at high speed and the safety buffer is too short".


3. **Object Permanence & Occlusion**:

    In a messy streetworks environment, equipment often obscures other critical safety features. For example, a utility van might temporarily block the view of a safety sign or a trench edge. The model must maintain object permanence—remembering that a hazard still exists and must be accounted for in the safety reasoning, even when it is not currently visible in the frame.

4. **Depth Perception from 2D Monocular Video**:

    Compliance rules are often measured in precise distances (e.g., "spoil heaps must be 0.5m from the trench edge"). Extracting accurate 3D spatial relationships from a 2D mobile phone video is notoriously difficult. The model must use its physical common sense to estimate depth and scale relative to known objects (like a traffic cone or a standard barrier) to validate these measurements.

5. **Temporal Alignment of Rules**:

    A "Decision Reel" requires perfect synchronization between the compliance logic and the video timestamps. If the model detects a trip hazard at 00:14 but the reasoning refers to a frame at 00:18, the audit trail loses legal and operational credibility. Ensuring the "Reasoning" engine is tightly coupled with the "World Model" video frames is a complex synchronization task.

6. **Evaluation Complexity**:

    Conflicting or interacting rules (e.g., pedestrian safety vs traffic flow) must be resolved holistically, not independently.

## **Implementation Scope**

Street-works compliance is a broad, multi-rule problem spanning dozens of safety conditions, contextual interpretations, and jurisdiction-specific regulations. Implementing full end-to-end compliance against standards such as the Red Book or MUTCD is **intentionally out of scope** for this recipe.

The objective of this work is not to build a fully-functional compliance agent, but to test a focused hypothesis:

> **Cosmos Reason-2 introduces a new dimension to visual analysis by enabling structured spatiotemporal reasoning over real-world inspection tasks.**

Rather than attempting to cover the full regulatory surface area, this implementation concentrates on a single, carefully selected compliance scenario:

**Barrier Continuity and Effectiveness**

The task is to determine whether a safety barrier that is visibly present is actually effective. In practice, barriers may be broken, disconnected, sagging, partially detached, or leave clear pedestrian access gaps despite appearing compliant in isolated frames.

This scenario is intentionally chosen because it cannot be reduced to simple object presence detection. A traditional pipeline might detect:
* "Barrier present"
* "Cone present"
* "Fence present"

But presence alone does not imply compliance.

Evaluating barrier effectiveness requires several reasoning dimensions:

* **Spatiotemporal reasoning**
    
    The camera moves through the scene, and gaps may only become evident across adjacent frames. Determining continuity requires integrating information over time rather than analysing a single image.

* **Physical continuity reasoning**

    The system must reason about connectedness, structural integrity, and whether barrier segments form an uninterrupted boundary.

* **Intent and failure-mode interpretation**

    A sagging chain, a detached segment, or a misaligned panel may still register as a “barrier” object, but functionally fail its protective intent. The task therefore involves interpreting whether the barrier achieves its safety purpose.

* **Contextual judgment under egocentric motion**

    The inspection video is captured in a walk-around format. The model must maintain consistency of interpretation despite changing viewpoint, scale, and partial occlusion.

Each detected discontinuity is emitted as a timestamped entry in the **Decision Reel**, paired with supporting video evidence and structured output suitable for downstream reporting. A **visual collage** and **structured summary** are generated to demonstrate how individual reasoning events can be composed into an audit-ready artefact.

*This narrow focus is deliberate. By deeply analysing a single, non-trivial compliance reasoning task, the implementation isolates and evaluates the added value of Cosmos Reason-2's structured visual reasoning capabilities. Extending the same pattern to additional compliance rules and regulatory standards is a natural next step, but remains outside the scope of this submission.*

## **Approach: Prompt Engineering**

We use prompt engineering as a lightweight form of domain encoding, positioning Cosmos Reason-2 as a virtual field inspector rather than a generic video summarizer.

Instead of custom training or fine-tuning, we translate compliance rules into structured system and user prompts that define inspection intent, visual criteria, and reporting format. The prompts explicitly articulate functional expectations — such as continuity of barriers, minimum separation distances, or structural integrity — and instruct the model to evaluate the scene against those constraints.

This approach demonstrates that targeted compliance use cases can be operationalised through careful prompt design alone, without the need for additional model training. It also allows rapid iteration: inspection logic can be modified, extended, or narrowed simply by adjusting rule descriptions within the prompt, making the system adaptable to evolving standards.

### **Sensitivity to Prompt Design**

During development, we observed that reasoning quality was highly sensitive to prompt structure and constraint formulation. Several trade-offs emerged:

1. **Reasoning overload vs. focus**

    Overly broad prompts (e.g., "analyse the scene for safety issues") resulted in generic descriptions rather than targeted compliance reasoning. Narrow, rule-specific intent statements produced more actionable outputs.

2. **Structured output vs. narrative richness**

    Strict JSON schemas improved consistency and downstream parsing, but overly rigid formatting occasionally suppressed intermediate reasoning signals. A balance was required between structured outputs and allowing sufficient reasoning latitude.

3. **Temporal instruction clarity**

    Explicitly instructing the model to reason across frames (e.g., "evaluate continuity over time") improved detection of barrier gaps compared to prompts framed around single-frame assessment.

4. **Functional framing vs. object detection framing**

    Prompts that emphasised functional effectiveness ("is the barrier preventing pedestrian access?") yielded better results than prompts framed around object presence ("is there a barrier present or a gap present?").

*These observations reinforce that prompt engineering functions as lightweight domain encoding. The effectiveness of Cosmos Reason-2 in compliance tasks depends not only on model capability, but on how inspection intent and evaluation criteria are articulated.*

## **Data Generation**

The goal of this recipe is to demonstrate the **reasoning capabilities of NVIDIA Cosmos Reason-2**, rather than to build or evaluate a production-grade compliance system. For the purposes of the Cookoff demo, we therefore use **synthetic, staged video data**.

The focus of the demo is to ensure that the video samples:
* are **egocentric** (handheld, walk-around perspective),
* contain **intentional camera motion**, and
* create **reasoning pressure** that requires spatial, temporal, and physical inference, rather than simple object detection.

The visual realism or regulatory completeness of the scenes is not critical; instead, the emphasis is on creating short video clips that expose the model to ambiguous, relational configurations that must be reasoned about across frames.

> No ground-truth labels or measurements are used; compliance judgments are derived solely from model reasoning over the video.

Video footage is generated for the selected reasoning example using simple, reproducible setups. The script is for illustrative purposes - we started with the high level framework described in the following  script, but then evolved iteratively based on the output.


### **Video Capture Scripts (Synthetic Data)**

<details>
<summary><strong>Example A: Barrier Continuity and Effectiveness</strong></summary>

**Objective**
Show that a safety barrier exists but is ineffective due to a visible gap or break.

**Setup (2 minutes)**
* 2 cones / chairs / posts
* A chain, rope, tape, or scarf between them
* Create an obvious gap:
    * chain not attached on one side, or
    * chain sagging on the ground, or
    * gap wide enough to walk through

**Recording Script (10-15 seconds)**

*Camera*: Mobile phone, handheld

*Movement*: Slow walk, slight arc

1. Start (0-4s): Frame both cones and the chain clearly.Make sure the gap is visible.
2. Approach (4-8s): Walk closer, lowering the camera slightly to show the chain’s condition.
3. Angle shift (8-12s): Walk past at an angle so the gap stays visible across frames.
4. End (12-15s): Brief pause centered on the gap.

**What Cosmos should reason about**

* Barrier intent (to block access)
* Continuity failure over time
* Gap persists despite camera movement

**Example intent prompt**

“Verify whether the pedestrian safety barrier is continuous and effective.”

## **Implementation**

### **Setup**

> This notebook is intended to run in Google Colab (GPU runtime recommended).
Colab-specific utilities are used for file upload and authentication.
Dependencies are listed in requirements.txt for reference.

#### Install Dependencies

In [ ]:
!pip install -q transformers>=4.57.0 torch accelerate pillow huggingface_hub

#### Define basic imports

In [ ]:
import os
import re
import cv2
import json
import math
import torch
import numpy as np
import psutil

from datetime import datetime

from google.colab import userdata, files
from huggingface_hub import login

import transformers
from transformers import Qwen3VLForConditionalGeneration, Qwen3VLProcessor, AutoProcessor

from PIL import Image
from IPython.display import Video, display
import matplotlib.pyplot as plt

#### Check the environment - Verify the GPU

In [ ]:
!nvidia-smi

Sat Feb 14 10:05:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

#### Ensure that CUDA is available

In [ ]:
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU! Go to Runtime → Change runtime type → T4 GPU")

GPU: Tesla T4 (15.6 GB)


#### Check if using High-RAM Memory
Select High-RAM in the Runtime if required

In [ ]:
ram_gb = psutil.virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 13.6 gigabytes of available RAM

Not using a high-RAM runtime


#### Get your Hugging Face Access Token

1. Setup a finegrained access token on [Hugging Face](https://huggingface.co/settings/tokens) and grant repository access to nvidia/Cosmos-Reason2-2B, selecting
    * Read access to contents of selected repos
    * View access requests for selected gated repos

2. Add to secrets to configure Colab to access Hugging Face resources.
3. Login to Hugging Face

In [ ]:
HF_TOKEN = userdata.get('HF_TOKEN')

In [ ]:
login(token=HF_TOKEN)
print("Logged in to Hugging Face")

Logged in to Hugging Face


### **Utilities**

#### Upload a file from the local device

In [ ]:
def upload_file():

    print("Upload a file:")
    uploaded = files.upload()

    if uploaded:
        # Get the filename of the first uploaded file
        video_path = list(uploaded.keys())[0]

        print(f"\nLoaded: {video_path}")
    else:
        print("No video uploaded")
        video_path = None
    return video_path

#### Functions to load/initialise NVIDIA Cosmos Reason-2B Model
Follow instructions from the [Cosmos Reason Cookbook](https://github.com/nvidia-cosmos/cosmos-reason2/blob/main/scripts/inference_sample.py)

In [ ]:
SEPARATOR = "-" * 20

PIXELS_PER_TOKEN = 32**2

In [ ]:
def load_model():

    # Ensure reproducibility
    transformers.set_seed(0)

    # load model
    MODEL_NAME = "nvidia/Cosmos-Reason2-2B"  # 2B fits on T4 (16GB)

    print(f"Loading {MODEL_NAME}...")
    print("(This takes 2-3 minutes on first run)")

    model = Qwen3VLForConditionalGeneration.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.float16,
        device_map="auto",
        attn_implementation="sdpa",
        trust_remote_code=True
    )
    processor = Qwen3VLProcessor.from_pretrained(MODEL_NAME)

    # Optional: Limit vision tokens
    min_vision_tokens = 256
    max_vision_tokens = 8192
    processor.image_processor.size = {
        "shortest_edge": min_vision_tokens * PIXELS_PER_TOKEN,
        "longest_edge": max_vision_tokens * PIXELS_PER_TOKEN,
    }
    processor.video_processor.size = {
        "shortest_edge": min_vision_tokens * PIXELS_PER_TOKEN,
        "longest_edge": max_vision_tokens * PIXELS_PER_TOKEN,
    }

    print(f"\nModel loaded!")
    print(f"GPU Memory Used: {torch.cuda.memory_allocated()/1e9:.1f} GB")
    return model, processor

#### Function to run inference on Cosmos Reason2
Follow instructions from the [Cosmos Reason Cookbook](https://github.com/nvidia-cosmos/cosmos-reason2/blob/main/scripts/inference_sample.py)

In [ ]:
def run_inference(model, processor, video_path):

    """
    Purpose:
      Executes a single reasoning pass over the input video using Cosmos Reason-2.

    Process:
    - The full video file is passed directly to the model endpoint.
    - Frame extraction and spatiotemporal reasoning are handled internally by the model.
    - The prompt defines the expected JSON schema for structured output.

    Returns:
      A JSON object containing:
        - gap_timestamps: list of timestamps where barrier discontinuities are detected
        - confidence: model-level confidence score for the inference
    """

    # Create inputs
    # IMPORTANT: Media is listed before text to match training inputs
    conversation = [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "video",
                    "video": video_path,
                },
                {"type": "text", "text": USER_PROMPT},
            ],
        },
    ]

    # Process inputs
    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        fps=8,
    )
    inputs = inputs.to(model.device)


    # Run inference
    generated_ids = model.generate(**inputs, max_new_tokens=4096)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids, strict=False)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

    response_text = output_text[0]

    print(SEPARATOR)
    print(response_text)
    print(SEPARATOR)

    # Parse and return extracted JSON
    try:
        json_data = extract_json_from_text(response_text)
        # Ensure required keys exist
        if "gap_timestamps" not in json_data or not isinstance(json_data["gap_timestamps"], list):
            json_data["gap_timestamps"] = []

        if "confidence" not in json_data or not isinstance(json_data["confidence"], str):
            json_data["confidence"] = "unknown"

        return json_data

    except Exception as e:
        print(f"JSON parsing failed for {video_path}: {e}")
        return {
            "gap_timestamps": [],
            "confidence": "low"
        }

#### Post Processing Pipeline

In [ ]:
def report_barrier_inspection(response, output_dir):
    """
    Post-processing pipeline

    Purpose:
      Converts raw model output into inspection-ready artefacts.

    Input:
      JSON response containing:
        - gap_timestamps
        - confidence

    Process:
      1. Merge adjacent or overlapping timestamps into continuous intervals.
      2. Extract representative evidence frames for each interval.
      3. Generate:
           - A visual collage of detected gaps
           - A decision reel video containing all flagged intervals
           - A structured summary report

    Outcome:
      Transforms frame-level detections into consolidated, audit-ready compliance evidence.
    """

    gap_timestamps = response["gap_timestamps"]
    confidence = response["confidence"]

    print(f'Gap timestamps: {gap_timestamps}')
    if not gap_timestamps:
        print("No discontinuities detected.")
        return response

    intervals = merge_timestamps(gap_timestamps)

    generate_report_bundle(
        video_path,
        intervals,
        output_dir=output_dir,
        fps=30
    )

    return response

#### Utilities for the post-processing pipeline

In [ ]:
def extract_json_from_text(text):
    """
    Extract the first valid JSON object from model output.
    Handles markdown fences and stray reasoning text.
    """
    # Remove markdown code fences if present
    text = text.replace("```json", "").replace("```", "").strip()

    # Extract first {...} block using regex
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        raise ValueError("No JSON object found in output.")

    json_str = match.group(0)

    return json.loads(json_str)

In [ ]:
def parse_timestamp_to_seconds(ts_str):
    """
    Convert 'mm:ss.ff' → seconds (float)
    """
    mins, secs = ts_str.split(":")
    return int(mins) * 60 + float(secs)

In [ ]:
def merge_timestamps(gap_timestamps, max_gap=0.7, padding=0.1):

    times = sorted([parse_timestamp_to_seconds(ts) for ts in gap_timestamps])

    if not times:
        return []

    intervals = []
    start = times[0]
    end = times[0]

    for t in times[1:]:
        if t - end <= max_gap:
            end = t
        else:
            intervals.append((start, end))
            start = t
            end = t

    intervals.append((start, end))

    # Apply SMALL padding only
    final = []
    for s, e in intervals:
        final.append((round(max(0, s - padding), 2),
                      round(e + padding, 2)))

    print(f'Intervals: {final}')
    return final

In [ ]:
def show_frames_from_timestamps(video_path, gap_timestamps):
    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Error opening video file.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    print(f"Video FPS: {fps}")

    for ts in gap_timestamps:
        seconds = parse_timestamp_to_seconds(ts)
        frame_number = int(seconds * fps)

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        ret, frame = cap.read()

        if not ret:
            print(f"Could not read frame at {ts}")
            continue

        # Convert BGR → RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(6,4))
        plt.imshow(frame_rgb)
        plt.title(f"Timestamp: {ts}  |  Frame: {frame_number}")
        plt.axis("off")
        plt.show()

    cap.release()

#### Generate Artefacts

In [ ]:
def create_decision_reel(video_path, intervals, output_path="decision_reel.mp4", pause_duration=0.6):

    """
    Generate a decision reel video containing all flagged intervals
    """
    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    intervals = sorted(intervals, key=lambda x: x[0])

    last_written_frame = -1
    segment_id = 1
    last_frame = None

    for start_sec, end_sec in intervals:

        start_frame = int(start_sec * fps)
        end_frame = int(end_sec * fps)

        if start_frame <= last_written_frame:
            start_frame = last_written_frame + 1

        if start_frame >= end_frame:
            continue

        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        label_main = f"GAP {segment_id} DETECTED"
        label_time = f"{start_sec:.1f}s - {end_sec:.1f}s"

        while True:
            current_frame = int(cap.get(cv2.CAP_PROP_POS_FRAMES))
            if current_frame > end_frame:
                break

            ret, frame = cap.read()
            if not ret:
                break

            # Draw translucent box
            overlay = frame.copy()
            cv2.rectangle(overlay, (30, 20), (520, 90), (0, 0, 0), -1)
            frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

            # Draw text
            cv2.putText(frame, label_main, (50, 55),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255), 2, cv2.LINE_AA)

            cv2.putText(frame, label_time, (50, 80),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2, cv2.LINE_AA)

            out.write(frame)
            last_written_frame = current_frame
            last_frame = frame.copy()

        # Pause at transition (freeze last frame)
        if last_frame is not None:
            pause_frames = int(fps * pause_duration)
            for _ in range(pause_frames):
                out.write(last_frame)

        segment_id += 1

    cap.release()
    out.release()

    print(f"Gap reel saved as: {output_path}")


In [ ]:
def create_evidence_collage(
    video_path,
    intervals,
    output_path="gap_collage.jpg",
    cols=3,
    add_header=True,
    title="Barrier Gap Detection Summary"
):
    """
    Generate a visual collage of detected gaps
    """

    cap = cv2.VideoCapture(video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    frames = []

    # Extract Midpoint Frames
    for i, (start_sec, end_sec) in enumerate(intervals):
        mid_sec = (start_sec + end_sec) / 2
        mid_frame = int(mid_sec * fps)

        cap.set(cv2.CAP_PROP_POS_FRAMES, mid_frame)
        ret, frame = cap.read()
        if not ret:
            continue

        # Draw translucent label box
        overlay = frame.copy()
        cv2.rectangle(overlay, (20, 15), (480, 85), (0, 0, 0), -1)
        frame = cv2.addWeighted(overlay, 0.4, frame, 0.6, 0)

        # Text
        cv2.putText(
            frame,
            f"GAP {i+1}",
            (40, 50),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.9,
            (0, 0, 255),
            2,
            cv2.LINE_AA,
        )

        cv2.putText(
            frame,
            f"{start_sec:.1f}s - {end_sec:.1f}s",
            (40, 78),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (255, 255, 255),
            2,
            cv2.LINE_AA,
        )

        frames.append(frame)

    cap.release()

    if not frames:
        print("No frames extracted.")
        return

    # Build Grid
    rows = math.ceil(len(frames) / cols)

    # Dark grey background instead of black
    collage = np.ones((rows * height, cols * width, 3), dtype=np.uint8) * 30

    for idx, frame in enumerate(frames):
        r = idx // cols
        c = idx % cols
        collage[r*height:(r+1)*height,
                c*width:(c+1)*width] = frame

    # Optional Header
    if add_header:
        header_height = 120
        final_image = np.ones(
            (collage.shape[0] + header_height, collage.shape[1], 3),
            dtype=np.uint8
        ) * 20

        final_image[header_height:, :] = collage

        cv2.putText(
            final_image,
            title,
            (40, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.2,
            (255, 255, 255),
            3,
            cv2.LINE_AA,
        )

        cv2.putText(
            final_image,
            f"Total Gaps Detected: {len(frames)}",
            (40, 105),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (180, 180, 180),
            2,
            cv2.LINE_AA,
        )

    else:
        final_image = collage

    cv2.imwrite(output_path, final_image)
    print(f"Collage saved as: {output_path}")


In [ ]:
def compute_metrics(intervals, output_path):
    durations = [end - start for start, end in intervals]

    metrics =  {
        "video_duration": round(intervals[-1][1], 2) if intervals else 0,
        "total_gaps": len(intervals),
        "total_gap_time": round(sum(durations), 2),
        "average_gap_duration": round(
            sum(durations) / len(durations), 2
        ) if durations else 0,
        "longest_gap": round(max(durations), 2) if durations else 0,
        "gaps": [
            {
                "id": i + 1,
                "start": round(start, 2),
                "end": round(end, 2),
                "duration": round(end - start, 2)
            }
            for i, (start, end) in enumerate(intervals)
        ]
    }

    # Save JSON
    with open(output_path, "w") as f:
        json.dump(metrics, f, indent=2)
    print(f"Metrics saved as: {output_path}")

    return metrics

In [ ]:
def generate_summary(metrics, output_path):
    """
    Generate a human-readable text summary of gap detection results.
    """

    from datetime import datetime


    with open(output_path, "w") as f:
        f.write("Barrier Gap Detection Summary\n")
        f.write("================================\n\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")

        f.write(f"Video Duration: {metrics['video_duration']:.2f}s\n")
        f.write(f"Total Gaps Detected: {metrics['total_gaps']}\n")
        f.write(f"Total Exposure Time: {metrics['total_gap_time']:.2f}s\n")
        f.write(f"Average Gap Duration: {metrics['average_gap_duration']:.2f}s\n")
        f.write(f"Longest Gap: {metrics['longest_gap']:.2f}s\n\n")

        f.write("Gap Intervals:\n")
        f.write("--------------------------------\n")

        for gap in metrics["gaps"]:
            f.write(
                f"Gap {gap['id']}: "
                f"{gap['start']:.2f}s - {gap['end']:.2f}s "
                f"(Duration: {gap['duration']:.2f}s)\n"
            )
        print(f"Summary saved as: {output_path}")

In [ ]:
def generate_report_bundle(
    video_path,
    intervals,
    output_dir,
    fps=30
):
    """
    Generate all report artefacts.
    """
    os.makedirs(output_dir, exist_ok=True)

    # Create visuals
    collage_path = os.path.join(output_dir, "evidence_collage.jpg")
    reel_path = os.path.join(output_dir, "decision_reel.mp4")
    metrics_path = os.path.join(output_dir, "metrics.json")
    summary_path = os.path.join(output_dir, "summary.txt")

    create_evidence_collage(video_path, intervals, collage_path)
    create_decision_reel(video_path, intervals, reel_path)

    # Compute metrics
    metrics = compute_metrics(intervals, metrics_path)

    # Generate summary text
    generate_summary(metrics, summary_path)

    print(f"Report bundle created in: {output_dir}")

### **Prompts**

#### Define System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are a Safety Inspector for StreetWorks.

Your task is to detect structural discontinuities in temporary construction barriers.

A discontinuity is defined as:
- A visible gap between adjacent barrier panels.
- A missing or detached connection between panels.
- A break in the continuous physical boundary formed by the barriers.

You must evaluate the entire video sequence before producing output.
Multiple discontinuities may exist.

Focus only on physical continuity between adjacent barrier units.
Ignore trench safety, signage, workers, or other site conditions.

Report distinct discontinuity events, not repeated detections of the same gap across adjacent frames.
"""

#### Define User Prompt

In [ ]:
USER_PROMPT = """
You are provided with a video of a construction site.

Task:

Evaluate the full video sequence, both spatially and temporally.

A discontinuity may only become evident when observing how adjacent barrier panels align over time.

For each distinct discontinuity event:
- Identify the timestamp at which the gap or structural break is most clearly visible.
- Do not list multiple timestamps for the same physical gap unless they represent separate locations.

Important:
- Review the entire video before responding.
- Multiple distinct discontinuities may exist.
- Only include clearly visible structural gaps.
- Do not speculate about potential weaknesses — include only observable breaks.

A continuous barrier must form an uninterrupted physical boundary preventing pedestrian passage.

Output Format:

Return JSON only in the following format:

{
  "gap_timestamps": ["00:01.25", "00:04.80"],
  "confidence": "high"
}

If no discontinuities are detected, return:

{
  "gap_timestamps": [],
  "confidence": "high"
}
"""

### **Run Analysis**

#### Initialise Cosmos Reason2-2B Model

In [ ]:
model, processor = load_model()

Loading nvidia/Cosmos-Reason2-2B...
(This takes 2-3 minutes on first run)


config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/626 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]


Model loaded!
GPU Memory Used: 4.9 GB


#### Load Video File from local device and display

In [ ]:
video_path = upload_file()

Upload a file:


Saving barriricade-with-muti-gaps.mp4 to barriricade-with-muti-gaps.mp4

Loaded: barriricade-with-muti-gaps.mp4


In [ ]:
if video_path:
    # Display the video with a set width (e.g., 600px)
    # embed=True allows the video to play directly in the notebook cell
    display(Video(video_path, width=600, embed=True))

#### **Run 1: Cold Context**
This is the first run after loading the model, results in fragmented detection.

In [ ]:
response = run_inference(model, processor, video_path)

--------------------
```json
{
  "gap_timestamps": [
    "00:00.10",
    "00:02.10",
    "00:03.10",
    "00:04.10",
    "00:06.40",
    "00:07.40",
    "00:08.10",
    "00:09.60",
    "00:10.60",
    "00:11.90",
    "00:13.90",
    "00:15.40"
  ],
  "confidence": "high"
}
```
--------------------


In [ ]:
report = report_barrier_inspection(response, "output_run_1")

Gap timestamps: ['00:00.10', '00:02.10', '00:03.10', '00:04.10', '00:06.40', '00:07.40', '00:08.10', '00:09.60', '00:10.60', '00:11.90', '00:13.90', '00:15.40']
Intervals: [(0, 0.2), (2.0, 2.2), (3.0, 3.2), (4.0, 4.2), (6.3, 6.5), (7.3, 8.2), (9.5, 9.7), (10.5, 10.7), (11.8, 12.0), (13.8, 14.0), (15.3, 15.5)]
Collage saved as: output_run_1/evidence_collage.jpg
Gap reel saved as: output_run_1/decision_reel.mp4
Metrics saved as: output_run_1/metrics.json
Summary saved as: output_run_1/summary.txt
Report bundle created in: output_run_1


#### Output Sample

In [ ]:
display(Image.open("output_run_1/evidence_collage.jpg"))

In [ ]:
display(Video("output_run_1/decision_reel.mp4", width=600, embed=True))

#### **Run 2: Repeat in same Colab Session**
Repeat the inference in the same session. This results in consolidated detection.

In [ ]:
response = run_inference(model, processor, video_path)

--------------------
```json
{
  "gap_timestamps": [
    "00:00.10",
    "00:08.60",
    "00:09.60",
    "00:14.60"
  ],
  "confidence": "high"
}
```
--------------------


In [ ]:
report = report_barrier_inspection(response, "output_run_2")

Gap timestamps: ['00:00.10', '00:08.60', '00:09.60', '00:14.60']
Intervals: [(0, 0.2), (8.5, 8.7), (9.5, 9.7), (14.5, 14.7)]
Collage saved as: output_run_2/evidence_collage.jpg
Gap reel saved as: output_run_2/decision_reel.mp4
Metrics saved as: output_run_2/metrics.json
Summary saved as: output_run_2/summary.txt
Report bundle created in: output_run_2


#### Output Sample

In [ ]:
display(Image.open("output_run_2/evidence_collage.jpg"))

In [ ]:
display(Video("output_run_2/decision_reel.mp4", width=600, embed=True))

> #### **Run Variability and Temporal Consolidation**
>
> Across repeated inference runs within the same session, output granularity occasionally varied. Initial runs tended to produce more fragmented gap timestamps (e.g., 10–11 detections), whereas subsequent runs yielded fewer, temporally consolidated intervals (e.g., 3–4 detections).
>
> This behavior suggests sensitivity to decoding dynamics and reasoning calibration rather than a change in underlying visual interpretation. Importantly, consolidated outputs more closely resemble how a human inspector would report continuous barrier failures—as grouped incidents rather than frame-level events.
>
> To ensure stable compliance reporting, interval merging and deterministic decoding settings were incorporated into the workflow.

In [ ]:
# Download output from run 1
!zip -r output_run_1.zip /content/output_run_1/
files.download('output_run_1.zip')

# Download output from run 2
!zip -r output_run_2.zip /content/output_run_2/
files.download('output_run_2.zip')

  adding: content/output_run_1/ (stored 0%)
  adding: content/output_run_1/metrics.json (deflated 79%)
  adding: content/output_run_1/evidence_collage.jpg (deflated 1%)
  adding: content/output_run_1/summary.txt (deflated 62%)
  adding: content/output_run_1/decision_reel.mp4 (deflated 1%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  adding: content/output_run_2/ (stored 0%)
  adding: content/output_run_2/metrics.json (deflated 67%)
  adding: content/output_run_2/evidence_collage.jpg (deflated 2%)
  adding: content/output_run_2/summary.txt (deflated 52%)
  adding: content/output_run_2/decision_reel.mp4 (deflated 1%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **OBSERVATIONS**

The barrier continuity scenario provided a controlled environment to evaluate Cosmos Reason-2’s spatiotemporal reasoning behaviour under realistic inspection conditions. The following observations summarise both model-level reasoning characteristics and pipeline-level operational learnings.

1. **Spatiotemporal Reasoning Improves Event-Level Detection**
    Barrier discontinuities often became fully visible only across adjacent frames as camera motion revealed alignment shifts or panel separations. Prompts explicitly instructing temporal evaluation produced more coherent discontinuity detection than prompts framed around isolated frame inspection.
    
    Without clear temporal guidance, outputs tended toward descriptive summaries rather than structured compliance events. Explicitly encoding temporal continuity as an inspection criterion materially improved detection relevance.

2. **Functional Framing Outperforms Object Presence Detection**

    Prompts framed around “barrier presence” frequently produced correct object identification but failed to distinguish between intact and ineffective barriers. Reframing the task around functional continuity—whether the barrier formed an uninterrupted physical boundary—materially improved discontinuity detection.

    This supports the hypothesis that compliance reasoning requires interpretation of structural intent, not merely object detection.

3. **Reasoning Overload Affects Detection Granularity**
    Increasing reasoning verbosity and instruction density occasionally reduced timestamp coverage. When prompted to produce detailed internal reasoning alongside structured output, the model sometimes reported fewer discontinuities than when prompted more narrowly.
    
    This suggests a trade-off between:
    * Depth of reasoning articulation
    * Exhaustiveness of event enumeration

    In practice, focused, constraint-driven prompts produced more complete detection lists than prompts encouraging extensive deliberation. This reinforces the importance of calibrated prompt design in operational compliance workflows.

4. **Temporal Aggregation Is Necessary for Operational Use**

    Raw timestamp outputs required post-processing to merge temporally adjacent detections into meaningful inspection intervals. This temporal consolidation step transformed model reasoning into operational artefacts suitable for review.

5. **Evidence Presentation Affects Interpretability**
    Two evidence formats were generated:
    * A short Decision Reel highlighting discontinuities over time.
    * A static annotated collage summarising detected gaps.

    In practice, the collage provided faster visual validation during review, while the reel better illustrated temporal emergence of discontinuities. This suggests that reasoning outputs benefit from multiple evidence representations depending on stakeholder needs.

## **Conclusion**

While this recipe demonstrates street works compliance, the same intent-driven reasoning pattern applies to worker safety, asset inspections, job audit and handover, retail operations, etc.

This implementation does not attempt to replicate full regulatory compliance for street-works inspection. Instead, it evaluates a focused hypothesis:

> Cosmos Reason-2 introduces a new dimension to visual analysis by enabling structured spatiotemporal reasoning over real-world inspection tasks.

Through a single, carefully scoped barrier continuity scenario, the study demonstrates that:

* The model can reason across temporal sequences rather than isolated frames.

* Functional integrity can be evaluated beyond simple object presence detection.

* Compliance logic can be encoded through structured prompt design without additional training.

* Model outputs can be transformed into audit-ready artefacts via temporal aggregation and evidence packaging.

At the same time, development revealed important calibration dynamics. Detection coverage and output granularity were sensitive to prompt structure and reasoning density. Overly verbose or instruction-heavy prompts occasionally reduced event completeness, indicating that reasoning bandwidth and constraint clarity must be balanced carefully.

These findings suggest that scalable compliance workflows are likely best implemented using modular, rule-specific reasoning passes, rather than a single monolithic prompt attempting to encode all regulatory conditions simultaneously. Focused inference per rule preserves reasoning clarity and allows structured aggregation of independent compliance checks.

Within its intentionally narrow scope, this implementation demonstrates that reasoning-oriented visual models can support inspection-style tasks that extend beyond conventional object detection pipelines. Extending the approach to additional rules, jurisdictions, and edge cases represents a natural next step, but the core result is clear: structured prompt-driven reasoning enables a qualitatively different mode of visual analysis for compliance applications.